In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import ta
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [63]:
tickers = ['TSLA','AAPL', 'MSFT', 'NVDA']
period = '1y'
interval = '1d'

In [60]:
def find_rsi_divergences_multi(tickers, rsi_period=14, lookback=20, period='1y', interval='1d'):
    """
    Detecta divergencias RSI para uno o varios tickers.
    
    Parámetros:
    - tickers: str o lista de str (ej. 'AAPL' o ['AAPL', 'MSFT'])
    - rsi_period: período del RSI (por defecto 14)
    - lookback: ventana para detectar extremos (por defecto 20)
    - period: período de descarga desde Yahoo Finance (por defecto '1y')
    - interval: intervalo temporal (por defecto '1d')
    
    Devuelve:
    - Diccionario con DataFrames por ticker que contienen las señales de compra/venta.
    """
    if isinstance(tickers, str):
        tickers = [tickers]

    results = {}

    for ticker in tickers:
        df = yf.download(ticker, period=period, interval=interval, multi_level_index=False)
        if df.empty:
            print(f"⚠️ No se pudieron descargar datos para {ticker}")
            continue

        df['RSI'] = ta.momentum.RSIIndicator(df['Close'], window=rsi_period).rsi()
        df['min_price'] = df['Close'].rolling(window=lookback, center=True).min()
        df['max_price'] = df['Close'].rolling(window=lookback, center=True).max()

        divergences = []
        for i in range(lookback, len(df) - lookback):
            precio_i    = df['Close'].iloc[i]
            min_price_i = df['min_price'].iloc[i]
            max_price_i = df['max_price'].iloc[i]
            rsi_i       = df['RSI'].iloc[i]

            # Bullish divergence → señal de compra
            if precio_i == min_price_i:
                prev_label = df['Close'][:i].idxmin()
                prev_pos   = df.index.get_loc(prev_label)
                if prev_pos < i and rsi_i > df['RSI'].iloc[prev_pos]:
                    divergences.append((df.index[i], '++COMPRAR'))

            # Bearish divergence → señal de venta
            if precio_i == max_price_i:
                prev_label = df['Close'][:i].idxmax()
                prev_pos   = df.index.get_loc(prev_label)
                if prev_pos < i and rsi_i < df['RSI'].iloc[prev_pos]:
                    divergences.append((df.index[i], '--VENDER'))

        results[ticker] = pd.DataFrame(divergences, columns=['Fecha', 'Señal'])

    return results

In [64]:
# tickers = ['AAPL', 'MSFT', 'NVDA']
divs_por_ticker = find_rsi_divergences_multi(tickers)

for ticker, df_divs in divs_por_ticker.items():
    print(f"\n📈 Divergencias para {ticker}:")
    print(df_divs)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


📈 Divergencias para TSLA:
        Fecha      Señal
0  2025-01-02  ++COMPRAR
1  2025-01-15   --VENDER
2  2025-03-25   --VENDER
3  2025-04-08  ++COMPRAR
4  2025-05-27   --VENDER
5  2025-06-05  ++COMPRAR
6  2025-06-23   --VENDER
7  2025-07-07  ++COMPRAR
8  2025-07-23   --VENDER
9  2025-08-01  ++COMPRAR
10 2025-08-21  ++COMPRAR

📈 Divergencias para AAPL:
        Fecha      Señal
0  2025-01-29   --VENDER
1  2025-02-24   --VENDER
2  2025-04-02   --VENDER
3  2025-05-01   --VENDER
4  2025-05-07  ++COMPRAR
5  2025-05-23  ++COMPRAR
6  2025-06-06   --VENDER
7  2025-06-17  ++COMPRAR
8  2025-07-03   --VENDER
9  2025-07-22   --VENDER
10 2025-08-01  ++COMPRAR
11 2025-08-13   --VENDER

📈 Divergencias para MSFT:
       Fecha      Señal
0 2024-11-21  ++COMPRAR
1 2025-01-14  ++COMPRAR
2 2025-01-28   --VENDER
3 2025-02-20   --VENDER
4 2025-03-13  ++COMPRAR
5 2025-03-25   --VENDER
6 2025-08-04   --VENDER

📈 Divergencias para NVDA:
       Fecha     Señal
0 2024-10-21  --VENDER
1 2025-01-06  --VENDER
2 2025

In [53]:
find_rsi_divergences_multi(['AAPL', 'MSFT', 'NVDA'])

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


{'AAPL':         Fecha      Señal
 0  2025-01-29   --VENDER
 1  2025-02-24   --VENDER
 2  2025-04-02   --VENDER
 3  2025-05-01   --VENDER
 4  2025-05-07  ++COMPRAR
 5  2025-05-23  ++COMPRAR
 6  2025-06-06   --VENDER
 7  2025-06-17  ++COMPRAR
 8  2025-07-03   --VENDER
 9  2025-07-22   --VENDER
 10 2025-08-01  ++COMPRAR
 11 2025-08-13   --VENDER,
 'MSFT':        Fecha      Señal
 0 2024-11-21  ++COMPRAR
 1 2025-01-14  ++COMPRAR
 2 2025-01-28   --VENDER
 3 2025-02-20   --VENDER
 4 2025-03-13  ++COMPRAR
 5 2025-03-25   --VENDER
 6 2025-08-04   --VENDER,
 'NVDA':        Fecha     Señal
 0 2024-10-21  --VENDER
 1 2025-01-06  --VENDER
 2 2025-01-23  --VENDER
 3 2025-02-20  --VENDER
 4 2025-04-09  --VENDER
 5 2025-08-12  --VENDER}

In [55]:
def find_rsi_divergences(df, rsi_period=14, lookback=20):
    # 1) RSI unidimensional
    df['RSI'] = ta.momentum.RSIIndicator(df['Close'], window=rsi_period).rsi()

    # 2) Ventanas para extremos
    df['min_price'] = df['Close'].rolling(window=lookback, center=True).min()
    df['max_price'] = df['Close'].rolling(window=lookback, center=True).max()

    divergences = []
    for i in range(lookback, len(df)-lookback):
        precio_i    = df['Close'].iloc[i]
        min_price_i = df['min_price'].iloc[i]
        max_price_i = df['max_price'].iloc[i]
        rsi_i       = df['RSI'].iloc[i]

        # — Bullish divergence —
        if precio_i == min_price_i:
            # etiqueta del mínimo previo
            prev_label = df['Close'][:i].idxmin()
            # posición entera de ese label
            prev_pos = df.index.get_loc(prev_label)
            if prev_pos < i and rsi_i > df['RSI'].iloc[prev_pos]:
                divergences.append((df.index[i], '++COMPRAR'))

        # — Bearish divergence —
        if precio_i == max_price_i:
            prev_label = df['Close'][:i].idxmax()
            prev_pos   = df.index.get_loc(prev_label)
            if prev_pos < i and rsi_i < df['RSI'].iloc[prev_pos]:
                divergences.append((df.index[i], '--VENDER'))

    return pd.DataFrame(divergences, columns=['Fecha', 'Señal'])


def detect_engulfing_patterns(tickers, period='1y', interval='1d', tolerance=0.005, strength_ratio=1.5):
    """
    Detecta patrones Compra y Venta Engulfing en uno o varios tickers.
    
    Parámetros:
    - tickers: str o lista de str (ej. 'AAPL' o ['AAPL', 'MSFT'])
    - period: período de descarga (por defecto '1y')
    - interval: intervalo temporal (por defecto '1d')
    - tolerance: tolerancia para comparar apertura y cierre (por defecto 0.5%)
    - strength_ratio: relación de amplitud entre velas (por defecto 1.5)
    
    Devuelve:
    - Diccionario con DataFrames por ticker que contienen las señales detectadas.
    """
    if isinstance(tickers, str):
        tickers = [tickers]

    results = {}

    for ticker in tickers:
        df = yf.download(ticker, period=period, interval=interval, multi_level_index=False)
        if df.empty:
            print(f"No se pudieron descargar datos para {ticker}")
            continue

        df["Date"] = pd.to_datetime(df.index)
        df.index.name = "time"

        # Clasificación de velas
        df["Candle way"] = np.where(df["Open"] < df["Close"], 1, -1)
        df["amplitude"] = np.abs(df["Close"] - df["Open"])

        # Inicializar columnas
        df["Compra Engulfing"] = np.nan
        df["Venta Engulfing"] = np.nan

        # Compra Engulfing
        df.loc[
            (df["Candle way"].shift(5) == -1) &
            (df["Candle way"].shift(4) == -1) &
            (df["Candle way"].shift(3) == -1) &
            (df["Candle way"].shift(2) == -1) &
            (df["Candle way"].shift(1) == -1) &
            (df["Candle way"] == 1) &
            (df["Close"].shift(1) < df["Open"] * (1 + tolerance)) &
            (df["Close"].shift(1) > df["Open"] * (1 - tolerance)) &
            (df["amplitude"].shift(1) * strength_ratio < df["amplitude"]),
            "Compra Engulfing"
        ] = 1

        # Venta Engulfing
        df.loc[
            (df["Candle way"].shift(5) == 1) &
            (df["Candle way"].shift(4) == 1) &
            (df["Candle way"].shift(3) == 1) &
            (df["Candle way"].shift(2) == 1) &
            (df["Candle way"].shift(1) == 1) &
            (df["Candle way"] == -1) &
            (df["Close"].shift(1) < df["Open"] * (1 + tolerance)) &
            (df["Close"].shift(1) > df["Open"] * (1 - tolerance)) &
            (df["amplitude"].shift(1) * strength_ratio < df["amplitude"]),
            "Venta Engulfing"
        ] = -1

        # Filtrar señales
        signals = df[df["Compra Engulfing"].notna() | df["Venta Engulfing"].notna()]
        results[ticker] = signals[["Date", "Open", "Close", "Compra Engulfing", "Venta Engulfing"]]

    for ticker, df_signals in results.items():
        if df_signals.empty:
            print(f"\nNo se detectaron señales para {ticker}.")
        else:
            print(f"\n📈 Señales para {ticker}:")
            print(df_signals[['Date', 'Compra Engulfing', 'Venta Engulfing']])
    # return results

In [ ]:
for ticker

In [56]:
# Detectar patrones en varias acciones
tickers = ['AAPL', 'MSFT', 'NVDA']
detect_engulfing_patterns(tickers)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


📈 Señales para AAPL:
                 Date  Compra Engulfing  Venta Engulfing
time                                                    
2024-12-05 2024-12-05               NaN             -1.0
2024-12-27 2024-12-27               NaN             -1.0
2025-01-06 2025-01-06               1.0              NaN

📈 Señales para MSFT:
                 Date  Compra Engulfing  Venta Engulfing
time                                                    
2025-02-05 2025-02-05               1.0              NaN
2025-02-21 2025-02-21               NaN             -1.0
2025-08-12 2025-08-12               1.0              NaN

No se detectaron señales para NVDA.


In [67]:
# Uso
ticker = 'NVDA'
df = yf.download(ticker, period='1y', interval='1d', multi_level_index=False)
divs = find_rsi_divergences(df)
divs

[*********************100%***********************]  1 of 1 completed


,Fecha,Señal
0,2024-10-21,--VENDER
1,2025-01-06,--VENDER
2,2025-01-23,--VENDER
3,2025-02-20,--VENDER
4,2025-04-09,--VENDER
5,2025-08-12,--VENDER
